# v18 · The Blackjack subgroup F₂₁ as an indexer: a negative result, with both halves measured
`VAPMIP/benchmarks/blackjack_vs_horner_bench.py` (git 2026-09-25). F₂₁ = {x ↦ ax + b mod 7 : a ∈ {1, 2, 4}}, the normaliser of a Sylow-7 subgroup of PSL(2,7).

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
bj = load("VAPMIP/benchmarks/blackjack_vs_horner_bench.py", "bj")
print(bj.verify_group())
c = bj.find_collision()
print("composing digits collapses to one of 21 states: sequences", c["seq_a"], "and", c["seq_b"], "both give", c["final_state"])
assert c and c["seq_a"] != c["seq_b"]

{'size': 21, 'closed': True, 'has_inverses': True}
composing digits collapses to one of 21 states: sequences (0,) and (0, 0) both give (0, 1, 2, 3, 4, 5, 6)


**Half 1 (composition).** A word of group elements composed into one element has at most 21 possible results, so it cannot be an injective address for arbitrarily many strings. The collision above is found by brute force, not argued.

**Half 2 (use the 21 elements as a digit alphabet).** Lossless again, since the digit sequence is kept. Cost against base 97:

In [3]:
rows = bj.bench_sizes(sizes=(140, 500, 1000, 2000))
print("chars | base21 digits bits  enc_ms  dec_ms | base97 digits bits  enc_ms  dec_ms | bits21/bits97")
for r in rows: print("%5d | %13d %5d %7.2f %7.2f | %13d %5d %7.2f %7.2f | %.2f" % (r["chars"], r["n_digits_21"], r["bits_21"], r["enc_ms_21"], r["dec_ms_21"], r["n_digits_97"], r["bits_97"], r["enc_ms_97"], r["dec_ms_97"], r["bits_21"]/r["bits_97"]))
print("log2(97)/log2(21) =", round(math.log2(97)/math.log2(21), 3))

chars | base21 digits bits  enc_ms  dec_ms | base97 digits bits  enc_ms  dec_ms | bits21/bits97
  140 |           255  1119    0.63    0.45 |           140   924    0.10    0.30 | 1.21
  500 |           911  3999    6.29    5.71 |           500  3300    0.44    2.71 | 1.21
 1000 |          1821  7999   23.44   20.88 |          1000  6600    1.47    9.04 | 1.21
 2000 |          3643 15999   41.66   31.17 |          2000 13200    1.67   13.96 | 1.21
log2(97)/log2(21) = 1.503


Base 21 needs about 1.2× the bits here (the byte-to-base-21 step works on UTF-8 bytes, 8 bits per character, against 6.6 for base 97) and more digit operations. A smaller alphabet never shortens an exact address; it only lengthens the digit string.